In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Imports

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import numpy as np
import random
import os
import glob
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.9.0+cu126
GPU available: True


## Seeding

In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)
print("Seeds locked!")

Seeds locked!


## Path

In [4]:
INPUT_BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH  = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')

## Generate Synthetic Dataset Q1 & Q2

In [5]:
def generate_synthetic_dataset(stems_dir, noise_dir, output_dir,
                                samples_per_genre=50, target_sr=22050, duration=30):
    genres = ["blues","classical","country","disco","hiphop",
              "jazz","metal","pop","reggae","rock"]
    target_length = target_sr * duration
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)

    for genre in genres:
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders:
            print(f"Warning: No songs for {genre}")
            continue

        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)
                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)

            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)

                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)

                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

    print("Dataset generation complete!")

seed_everything(42)
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)

# Q1 Count the files
wav_files = glob.glob(os.path.join(OUTPUT_PATH, '**', '*.wav'), recursive=True)
print(f"Q1 — Total .wav files: {len(wav_files)}")  # Expected: 500

# Q2 Tensor shape
sample_wav = wav_files[0]
waveform, sr = torchaudio.load(sample_wav)
print(f"Q2 — Waveform shape: {tuple(waveform.shape)}")  # Expected: (1, 661500)

Dataset generation complete!
Q1 — Total .wav files: 500
Q2 — Waveform shape: (2, 661500)


## Extract Features Q3

In [6]:
def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)

    for wav_path in wav_files:
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        out_path.parent.mkdir(parents=True, exist_ok=True)

        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        torch.save(mel_spec_db, out_path)

    print(f"Saved {len(wav_files)} feature files.")

FEATURES_DIR = '/kaggle/working/features/train'
extract_and_save_features(OUTPUT_PATH, FEATURES_DIR)

# Q3 Answer — shape of saved tensor
pt_files = glob.glob(os.path.join(FEATURES_DIR, '**', '*.pt'), recursive=True)
sample_tensor = torch.load(pt_files[0])
print(f"Q3 — Feature tensor shape: {tuple(sample_tensor.shape)}")  # Expected: (1, 128, 1293)

Saved 500 feature files.
Q3 — Feature tensor shape: (2, 128, 1292)


##  Build the CRNN Model Q4 & Q5

In [7]:
class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # CNN Backbone
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # RNN: input_size = 64 channels * 32 mel_bins = 2048
        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Classifier: bidirectional → hidden_size * 2 = 128
        self.fc = nn.Linear(64 * 2, num_classes)

    def forward(self, x):
        # x shape: (Batch, 1, 128, Time)

        # Step 1: CNN
        x = self.cnn(x)
        # Shape: (Batch, 64, 32, Time)

        # Step 2: Bridge — reshape for LSTM
        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2)   # → (Batch, Time, 64, 32)
        x = x.reshape(b, t, c * f)  # → (Batch, Time, 2048)

        # Step 3: LSTM
        x, _ = self.lstm(x)          # → (Batch, Time, 128)

        # Step 4: Global Max Pooling over time
        x, _ = torch.max(x, dim=1)  # → (Batch, 128)

        # Step 5: Classifier
        logits = self.fc(x)          # → (Batch, 10)
        return logits


# Q4 — Shape after second MaxPool2d
dummy = torch.randn(32, 1, 128, 1293)
model_test = CRNN()
after_cnn = model_test.cnn(dummy)
print(f"Q4 — Shape after 2nd MaxPool2d: {tuple(after_cnn.shape)}")  # (32, 64, 32, 323)

# Q5 — LSTM trainable parameters
lstm_params = sum(p.numel() for p in model_test.lstm.parameters() if p.requires_grad)
print(f"Q5 — LSTM trainable parameters: {lstm_params}")  # 1081856

Q4 — Shape after 2nd MaxPool2d: (32, 64, 32, 323)
Q5 — LSTM trainable parameters: 1082368


## Dataset & DataLoader

In [11]:
class PrecomputedFeatureDataset(Dataset):
    def __init__(self, features_dir):
        self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)
        self.genres = sorted(['blues','classical','country','disco','hiphop',
                              'jazz','metal','pop','reggae','rock'])
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        genre = Path(file_path).parent.name
        label = self.genre_to_idx[genre]
        feature = torch.load(file_path)
        
        # Convert stereo (2, 128, T) to mono (1, 128, T) by averaging channels
        if feature.shape[0] == 2:
            feature = feature.mean(dim=0, keepdim=True)
        
        return feature, label

# Split into train/val (80/20)
full_dataset = PrecomputedFeatureDataset(FEATURES_DIR)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

Train samples: 400, Val samples: 100


## Train the model

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = CRNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

for epoch in range(num_epochs):
    # --- Training ---
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)

    # --- Validation ---
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for features, labels in val_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Train Acc: {100*train_correct/train_total:.2f}% | "
          f"Val Loss: {val_loss/len(val_loader):.4f} | "
          f"Val Acc: {100*val_correct/val_total:.2f}%")

Using device: cuda
Epoch [1/10] Train Loss: 2.2103 | Train Acc: 20.50% | Val Loss: 2.0742 | Val Acc: 25.00%
Epoch [2/10] Train Loss: 1.9837 | Train Acc: 41.50% | Val Loss: 1.9112 | Val Acc: 43.00%
Epoch [3/10] Train Loss: 1.8577 | Train Acc: 46.50% | Val Loss: 1.8307 | Val Acc: 44.00%
Epoch [4/10] Train Loss: 1.7081 | Train Acc: 51.25% | Val Loss: 1.7318 | Val Acc: 48.00%
Epoch [5/10] Train Loss: 1.5886 | Train Acc: 60.25% | Val Loss: 1.5880 | Val Acc: 56.00%
Epoch [6/10] Train Loss: 1.4574 | Train Acc: 67.50% | Val Loss: 1.4752 | Val Acc: 59.00%
Epoch [7/10] Train Loss: 1.3572 | Train Acc: 69.50% | Val Loss: 1.4500 | Val Acc: 55.00%
Epoch [8/10] Train Loss: 1.2377 | Train Acc: 75.25% | Val Loss: 1.3377 | Val Acc: 72.00%
Epoch [9/10] Train Loss: 1.1810 | Train Acc: 78.75% | Val Loss: 1.2893 | Val Acc: 68.00%
Epoch [10/10] Train Loss: 1.0539 | Train Acc: 80.75% | Val Loss: 1.1944 | Val Acc: 69.00%


In [13]:
torch.save(model.state_dict(), '/kaggle/working/crnn_model.pth')
print("Model saved!")

Model saved!
